# Challenge 4: Web Scraping y pipeline de datos.

- **Mini pipeline:** scraping de Books to Scrape → enriquecimiento de autores con una API externa → persistencia en PostgreSQL → consultas → indexación y medición de performance.

## 1. Configuración y conexión a PostgreSQL.

### Conexión a PostgreSQL y verificación de respuesta.
- **Entrada:** credenciales del archivo .env (no se escriben en el código).
- **Salida:** imprime (1,) si la conexión funciona.
- **Entregable que cubre:** prerrequisito de la base de datos (conexión).

In [1]:
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv(override=True)

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

cur = conn.cursor()
cur.execute("SELECT 1;")
print(cur.fetchone())
cur.close()
conn.close()

(1,)


## 2. Scraping de categorías y libros.

### 2.1. Importación de librerías y definición de funciones auxiliares del scraping.
- **Entrada:** ninguna. 
- **Salida:** constantes y funciones listas para usar.
- **Entregable que cubre:** scraping completo (requisito 3).

In [2]:
import re                         # expresiones regulares (limpiar el precio)
import time                       # pausas entre peticiones
import requests                   # peticiones HTTP
from bs4 import BeautifulSoup     # parseo del HTML
from urllib.parse import urljoin  # URLs absolutas a partir de relativas

BASE = "https://books.toscrape.com/"
HEADERS = {"User-Agent": "challenge4-student-project"}
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def get_soup(url):
    resp = requests.get(url, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    resp.encoding = "utf-8"
    return BeautifulSoup(resp.text, "html.parser")

def parse_price(text):
    return float(re.sub(r"[^\d.]", "", text))

### 2.2. Obtención de las categorías y recorrida de todos los libros con su paginación.
- **Entrada:** URL base del sitio.
- **Salida:** lista de diccionarios, uno por libro.
- **Entregable que cubre:** scraping completo (requisito 3).

In [3]:
def get_categories():
    soup = get_soup(BASE + "index.html")
    links = soup.select("div.side_categories ul li ul li a")
    return [(a.get_text(strip=True), urljoin(BASE, a["href"])) for a in links]

def scrape_category(name, url):
    books = []
    while url:
        soup = get_soup(url)
        for pod in soup.select("article.product_pod"):
            a = pod.select_one("h3 a")
            rating_word = pod.select_one("p.star-rating")["class"][1]
            books.append({
                "title": a["title"],
                "price": parse_price(pod.select_one("p.price_color").get_text()),
                "rating": RATING_MAP[rating_word],
                "category": name,
                "url": urljoin(url, a["href"]),
            })
        nxt = soup.select_one("li.next a")
        url = urljoin(url, nxt["href"]) if nxt else None
        time.sleep(0.2)
    return books

### 2.3. Ejecución del scraping completo y verificación de que no falta nada.
- **Salida esperada:** 50 categorías, 1000 libros y 1000 URLs únicas.

In [4]:
all_books = []
categories = get_categories()
print("categorías:", len(categories))
for name, url in categories:
    cat_books = scrape_category(name, url)
    print(f"{name}: {len(cat_books)}")
    all_books.extend(cat_books)

print("TOTAL:", len(all_books))
print("URLs únicas:", len({b["url"] for b in all_books}))

categorías: 50
Travel: 11
Mystery: 32
Historical Fiction: 26
Sequential Art: 75
Classics: 19
Philosophy: 11
Romance: 35
Womens Fiction: 17
Fiction: 65
Childrens: 29
Religion: 7
Nonfiction: 110
Music: 13
Default: 152
Science Fiction: 16
Sports and Games: 5
Add a comment: 67
Fantasy: 48
New Adult: 6
Young Adult: 54
Science: 14
Poetry: 19
Paranormal: 1
Art: 8
Psychology: 7
Autobiography: 9
Parenting: 1
Adult Fiction: 1
Humor: 10
Horror: 17
History: 18
Food and Drink: 30
Christian Fiction: 6
Business: 12
Biography: 5
Thriller: 11
Contemporary: 3
Spirituality: 6
Academic: 1
Self Help: 5
Historical: 2
Christian: 3
Suspense: 1
Short Stories: 1
Novels: 1
Health: 4
Politics: 3
Cultural: 1
Erotica: 1
Crime: 1
TOTAL: 1000
URLs únicas: 1000


## 3. Enriquecimiento de autores con API externa

### 3.1. Cache y estructuras de trabajo
- **Entrada:** nada (usa las librerías ya importadas)
- **Salida:** sesión con cache en disco, `api_log`, `authors_cache`, `book_author_map`
- **Entregable que cubre:** cache de respuestas de la API

In [5]:
import requests_cache
from collections import Counter
from urllib.parse import quote

SESSION = requests_cache.CachedSession("api_cache")

api_log = []
authors_cache = {}
book_author_map = {}

### 3.2 Función de llamada con manejo de errores
- **Entrada:** URL y parámetros opcionales
- **Salida:** `(json, None)` si salió bien, `(None, motivo)` si falló
- **Entregable que cubre:** manejo de 404, 429 y timeout; rate limiting

In [6]:
def log_case(subject, step, reason):
    api_log.append({"subject": subject, "step": step, "reason": reason})

def api_get(url, params=None, retries=3):
    for attempt in range(retries):
        try:
            resp = SESSION.get(url, params=params, headers=HEADERS, timeout=10)
        except requests.exceptions.Timeout:
            time.sleep(attempt + 1)
            continue
        except requests.exceptions.RequestException as e:
            return None, f"error de red ({type(e).__name__})"
        if resp.status_code == 200:
            if not resp.from_cache:
                time.sleep(0.3)
            try:
                return resp.json(), None
            except ValueError:
                return None, "respuesta que no es JSON"
        if resp.status_code == 404:
            return None, "404"
        if resp.status_code == 429:
            time.sleep(2 ** (attempt + 1))
            continue
        return None, f"HTTP {resp.status_code}"
    return None, "reintentos agotados"

### 3.3 Resolver el autor a partir del título
- **Entrada:** título del libro (de Books to Scrape)
- **Salida:** `(id_autor, nombre)` o `(None, None)` si no hay coincidencia
- **Entregable que cubre:** enriquecimiento por API externa; autor no encontrado → NULL + log

In [7]:
def clean_title(title):
    title = re.sub(r"\(.*?\)", "", title)
    title = re.split(r":| - ", title)[0]
    return re.sub(r"[^a-z0-9]+", " ", title.lower()).strip()

def resolve_author(title):
    data, err = api_get(
        "https://openlibrary.org/search.json",
        params={"title": clean_title(title), "limit": 5,
                "fields": "title,author_name,author_key"},
    )
    if err:
        log_case(title, "búsqueda por título", err)
        return None, None
    for doc in data.get("docs", []):
        if (clean_title(doc.get("title", "")) == clean_title(title)
                and doc.get("author_key") and doc.get("author_name")):
            return doc["author_key"][0], doc["author_name"][0]
    log_case(title, "búsqueda por título", "sin coincidencia exacta de título")
    return None, None

### 3.4 País del autor desde Wikipedia
- **Entrada:** nombre del autor
- **Salida:** país en español, o `None` si no se puede determinar
- **Entregable que cubre:** campo país necesario para la consulta obligatoria

In [8]:
COUNTRY_PATTERNS = [
    (r"(?<!south )(?<!latin )(?<!central )\bamerican\b", "Estados Unidos"),
    (r"\b(?:british|english|scottish|welsh)\b", "Reino Unido"),
    (r"\birish\b", "Irlanda"),
    (r"\bcanadian\b", "Canadá"),
    (r"\baustralian\b", "Australia"),
    (r"\bnew zealand(?:er)?\b", "Nueva Zelanda"),
    (r"\bfrench\b", "Francia"),
    (r"\bgerman\b", "Alemania"),
    (r"\bitalian\b", "Italia"),
    (r"\bspanish\b", "España"),
    (r"\bportuguese\b", "Portugal"),
    (r"\bdutch\b", "Países Bajos"),
    (r"\bbelgian\b", "Bélgica"),
    (r"\bswiss\b", "Suiza"),
    (r"\bswedish\b", "Suecia"),
    (r"\bnorwegian\b", "Noruega"),
    (r"\bdanish\b", "Dinamarca"),
    (r"\bpolish\b", "Polonia"),
    (r"\brussian\b", "Rusia"),
    (r"\bgreek\b", "Grecia"),
    (r"\bturkish\b", "Turquía"),
    (r"\bisraeli\b", "Israel"),
    (r"\bsouth african\b", "Sudáfrica"),
    (r"\bnigerian\b", "Nigeria"),
    (r"\bindian\b", "India"),
    (r"\bchinese\b", "China"),
    (r"\bjapanese\b", "Japón"),
    (r"\bmexican\b", "México"),
    (r"\bargentin(?:e|ian)\b", "Argentina"),
    (r"\bbrazilian\b", "Brasil"),
    (r"\bchilean\b", "Chile"),
    (r"\bcolombian\b", "Colombia"),
    (r"\bcuban\b", "Cuba"),
]
COUNTRY_REGEX = [(re.compile(p, re.I), c) for p, c in COUNTRY_PATTERNS]

def find_country(text):
    best = None
    for pattern, country in COUNTRY_REGEX:
        m = pattern.search(text)
        if m and (best is None or m.start() < best[0]):
            best = (m.start(), country)
    return best[1] if best else None

def fetch_country(name):
    page = quote(name.strip().rstrip(".").replace(" ", "_"), safe="")
    data, err = api_get(f"https://en.wikipedia.org/api/rest_v1/page/summary/{page}")
    if err:
        log_case(name, "Wikipedia", err)
        return None
    if data.get("type") == "disambiguation":
        log_case(name, "Wikipedia", "página de desambiguación")
        return None
    country = find_country(data.get("description") or "")
    if country is None:
        country = find_country((data.get("extract") or "").split(". ")[0])
    if country is None:
        log_case(name, "Wikipedia", "no se encontró nacionalidad en el texto")
    return country

### 3.5 Ficha del autor y enriquecimiento de todos los libros
- **Entrada:** lista `all_books`
- **Salida:** `authors_cache` (fichas por autor) y `book_author_map` (libro → autor)
- **Entregable que cubre:** enriquecimiento con ≥2 datos nuevos, cache, idempotencia

In [9]:
def build_author(key, name):
    ficha = {"name": name, "birth_year": None, "country": None,
             "external_api_id": key, "total_known_works": None,
             "api_source": "Open Library"}
    data, err = api_get(f"https://openlibrary.org/authors/{key}.json")
    if err:
        log_case(name, "Open Library autor", err)
    else:
        m = re.search(r"\b(\d{4})\b", str(data.get("birth_date", "")))
        if m:
            ficha["birth_year"] = int(m.group(1))
    works, err = api_get(f"https://openlibrary.org/authors/{key}/works.json",
                         params={"limit": 1})
    if err:
        log_case(name, "Open Library obras", err)
    else:
        ficha["total_known_works"] = works.get("size")
    ficha["country"] = fetch_country(name)
    if ficha["country"]:
        ficha["api_source"] = "Open Library + Wikipedia"
    return ficha

def run_enrichment(books):
    pending = [b for b in books if b["url"] not in book_author_map]
    for i, book in enumerate(pending, 1):
        key, name = resolve_author(book["title"])
        if key and key not in authors_cache:
            authors_cache[key] = build_author(key, name)
        book_author_map[book["url"]] = key
        if i % 50 == 0:
            print(f"  {i}/{len(pending)} libros procesados")
    print("listo:", len(book_author_map), "libros,", len(authors_cache), "autores")

### 3.6 Ejecución y verificación
- **Entrada:** `all_books`
- **Salida:** resumen, muestra de fichas y motivos del log
- **Entregable que cubre:** evidencia de que los datos vienen de la API y del manejo de errores

In [10]:
run_enrichment(all_books)

  50/1000 libros procesados
  100/1000 libros procesados
  150/1000 libros procesados
  200/1000 libros procesados
  250/1000 libros procesados
  300/1000 libros procesados
  350/1000 libros procesados
  400/1000 libros procesados
  450/1000 libros procesados
  500/1000 libros procesados
  550/1000 libros procesados
  600/1000 libros procesados
  650/1000 libros procesados
  700/1000 libros procesados
  750/1000 libros procesados
  800/1000 libros procesados
  850/1000 libros procesados
  900/1000 libros procesados
  950/1000 libros procesados
  1000/1000 libros procesados
listo: 1000 libros, 704 autores


In [11]:
con_autor = sum(1 for v in book_author_map.values() if v)
print("libros con autor:", con_autor, "| sin autor (NULL):", len(book_author_map) - con_autor)
print("autores:", len(authors_cache), "| con país:", sum(1 for a in authors_cache.values() if a["country"]))
print("casos en el log:", len(api_log))
print()
for ficha in list(authors_cache.values())[:5]:
    print(ficha)
print()
print(Counter(c["reason"] for c in api_log))

libros con autor: 837 | sin autor (NULL): 163
autores: 704 | con país: 415
casos en el log: 454

{'name': 'Rolf Potts', 'birth_year': None, 'country': 'Estados Unidos', 'external_api_id': 'OL2839167A', 'total_known_works': 10, 'api_source': 'Open Library + Wikipedia'}
{'name': 'Frances Mayes', 'birth_year': 1940, 'country': 'Estados Unidos', 'external_api_id': 'OL20984A', 'total_known_works': 47, 'api_source': 'Open Library + Wikipedia'}
{'name': 'Marilyn Brant', 'birth_year': None, 'country': None, 'external_api_id': 'OL6792839A', 'total_known_works': 14, 'api_source': 'Open Library'}
{'name': 'Paul Theroux', 'birth_year': 1941, 'country': 'Estados Unidos', 'external_api_id': 'OL4416468A', 'total_known_works': 266, 'api_source': 'Open Library + Wikipedia'}
{'name': 'Peter Mayle', 'birth_year': 1939, 'country': 'Reino Unido', 'external_api_id': 'OL29023A', 'total_known_works': 240, 'api_source': 'Open Library + Wikipedia'}

Counter({'sin coincidencia exacta de título': 163, '404': 157,